# S11 – Scorecard
Logistic regression on the 9 WoE features, then points scaling. Fitted on train (131,366 loans), compared on validation (43,790). Preparation and binning come from src/model and match S10 exactly (tests/test_binning.py).

## 1. Prepare, bin and convert to WoE

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # repo root, so src.model imports work

In [ ]:
import numpy as np
import pandas as pd

from src.model.binning import fit_binners, to_woe
from src.model.prepare import get_engine, load_config, prepare_sets

pd.set_option("display.max_columns", None)

In [ ]:
cfg = load_config()
train, validation, fills = prepare_sets(get_engine(), cfg)
binners = fit_binners(train, cfg)  # train only
woe_train = to_woe(binners, train, cfg, "train")
woe_val = to_woe(binners, validation, cfg, "validation")
features = list(binners)
print(woe_train.shape, woe_val.shape)

## 2. Overlap between the WoE features

In [ ]:
corr = woe_train[features].corr()  # train only
print(corr.round(2).to_string())

In [ ]:
upper = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
pairs = upper.stack().dropna()
print(pairs[pairs.abs() >= 0.3].sort_values(key=abs, ascending=False).round(2))

In [ ]:
# VIF: how much of each feature the other 8 can guess. A maths shortcut that
# gives the same numbers as running one regression per feature.
vif = pd.Series(np.linalg.inv(corr.to_numpy()).diagonal(), index=features)
print(vif.sort_values(ascending=False).round(2).to_string())

## 3. MI vs LTV

In [ ]:
ltv_bin = binners["original_ltv"].transform(train["original_ltv"], metric="bins")
mi_bin = binners["mi_percentage"].transform(train["mi_percentage"], metric="bins")
print(pd.crosstab(ltv_bin, mi_bin, rownames=["LTV"], colnames=["MI"]))
print(
    pd.crosstab(
        ltv_bin, mi_bin, values=train["target"], aggfunc="mean",
        rownames=["LTV"], colnames=["MI"],
    ).round(4)
)